<a href="https://colab.research.google.com/github/Shreya59422/Gen-AI_Lab/blob/main/GenAI_exp7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install pypdf sentence-transformers faiss-cpu transformers accelerate

In [ ]:
from google.colab import files

uploaded = files.upload()

pdf_path = next(
    name for name in uploaded
    if name.lower().endswith(".pdf")
)

print("Uploaded:", pdf_path)

Saving DLNN paper 1.pdf to DLNN paper 1 (1).pdf
Uploaded: DLNN paper 1 (1).pdf


In [ ]:
from pypdf import PdfReader

reader = PdfReader(pdf_path)

pages = []

for page_num, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""

    if text.strip():
        pages.append({
            "page": page_num,
            "text": text
        })

if not pages:
    raise ValueError(
        "No text extracted. Try a text-based PDF."
    )

chunks = []

chunk_size = 150
overlap = 30

for page in pages:
    words = page["text"].split()

    for start in range(0, len(words), chunk_size - overlap):
        chunk_text = " ".join(
            words[start:start + chunk_size]
        )

        if chunk_text.strip():
            chunks.append({
                "text": chunk_text,
                "page": page["page"]
            })

print("Pages extracted:", len(pages))
print("Chunks created:", len(chunks))
print("\nFirst chunk:\n", chunks[0]["text"])

Pages extracted: 8
Chunks created: 63

First chunk:
 Contents lists available at ScienceDirect Speech Communication journal homepage: www.elsevier.com/locate/specom Cross-attention fusion for audio-visual emotion recognition with shared transformer Jianjun Lei a ,∗, Kai Ye a, Ying Wang b a School of Computer Science and Technology, Chongqing University of Posts and Telecommunications, 400065 Chongqing, China b School of Artificial Intelligence, Chongqing University of Posts and Telecommunications, 400065 Chongqing, China A R T I C L E I N F O Keywords: Multimodal emotion recognition Transformer Cross-attention mechanism Feature fusion A B S T R A C T Accurately recognizing emotional states can facilitate achieving a natural human–computer interaction. However, conventional multimodal emotion recognition approaches usually obtain low recognition accuracy due to their inadequate integration of multiple modal features. This paper proposes an innovative multimodal emotion recognition frame

In [ ]:
import faiss
import numpy as np

from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    texts,
    normalize_embeddings=True,
    convert_to_numpy=True
).astype("float32")

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print("Embeddings shape:", embeddings.shape)
print("FAISS index size:", index.ntotal)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings shape: (63, 384)
FAISS index size: 63


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
)

device = "cuda" if torch.cuda.is_available() else "cpu"
llm = llm.to(device)

print("Language model loaded on:", device)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Language model loaded on: cpu


In [ ]:
def ask_rag(question, top_k=3):
    # Retrieve the question's embedding
    query_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32")

    # Search for relevant chunks
    k = min(top_k, len(chunks))
    scores, indices = index.search(query_embedding, k)

    retrieved = []

    for score, idx in zip(scores[0], indices[0]):
        if idx >= 0:
            retrieved.append({
                **chunks[int(idx)],
                "score": float(score)
            })

    # Build the context
    context = "\n\n".join(
        f"Page {item['page']}: {item['text']}"
        for item in retrieved
    )

    # Create the prompt
    prompt = f"""
Answer the question using only the context below.
If the answer is not present, say you cannot find it.

Context:
{context}

Question: {question}

Answer:
"""

    # Generate the answer
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        output = llm.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False
        )

    answer = tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    print("\nQuestion:", question)
    print("\nAnswer:", answer)
    print("\nRetrieved source pages:")

    for item in retrieved:
        print(
            f"Page {item['page']} "
            f"(similarity: {item['score']:.3f})"
        )

    return answer

In [ ]:
ask_rag("What is the main topic of this document?")


Question: What is the main topic of this document?

Answer: CRediT

Retrieved source pages:
Page 7 (similarity: 0.175)
Page 7 (similarity: 0.149)
Page 6 (similarity: 0.147)


'CRediT'

In [ ]:
ask_rag("Explain the main concepts discussed in the document.")


Question: Explain the main concepts discussed in the document.

Answer: key and value

Retrieved source pages:
Page 4 (similarity: 0.234)
Page 4 (similarity: 0.220)
Page 6 (similarity: 0.204)


'key and value'

In [ ]:
ask_rag("What are the advantages mentioned in the document?")


Question: What are the advantages mentioned in the document?

Answer: key and value

Retrieved source pages:
Page 4 (similarity: 0.191)
Page 4 (similarity: 0.188)
Page 6 (similarity: 0.172)


'key and value'